In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("nassau_candy.csv")

print("Shape:", df.shape)
print(df.dtypes)

Shape: (10194, 18)
Row ID              int64
Order ID              str
Order Date            str
Ship Date             str
Ship Mode             str
Customer ID         int64
Country/Region        str
City                  str
State/Province        str
Postal Code           str
Division              str
Region                str
Product ID            str
Product Name          str
Sales             float64
Units               int64
Gross Profit      float64
Cost              float64
dtype: object


In [2]:
# Preserve row fields
df["Order_Date_Raw"] = df["Order Date"]
df["Ship_Date_Raw"] = df["Ship Date"]

In [3]:
df["Order_Date"] = pd.to_datetime(
    df["Order_Date_Raw"],
    dayfirst=True,
    errors="coerce"
)

df["Ship_Date"] = pd.to_datetime(
    df["Ship_Date_Raw"],
    dayfirst=True,
    errors="coerce"
)

In [4]:
print("Invalid order dates:", df["Order_Date"].isna().sum())
print("Invalid ship dates:", df["Ship_Date"].isna().sum())

Invalid order dates: 0
Invalid ship dates: 0


In [5]:
# Create route identifier
print(df.columns.tolist())

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name', 'Sales', 'Units', 'Gross Profit', 'Cost', 'Order_Date_Raw', 'Ship_Date_Raw', 'Order_Date', 'Ship_Date']


In [12]:
# Create a destination identifier using existing columns
df["Route_State"] = (
    df["Country/Region"].astype("string").str.strip()
    + " -> "
    + df["State/Province"].astype("string").str.strip()
)

# Verify the result
display(
    df[
        ["Country/Region", "State/Province", "Route_State"]
    ].head(10)
)

,Country/Region,State/Province,Route_State
0,United States,Texas,United States -> Texas
1,United States,Illinois,United States -> Illinois
2,United States,Illinois,United States -> Illinois
3,United States,Illinois,United States -> Illinois
4,United States,Pennsylvania,United States -> Pennsylvania
5,United States,Kentucky,United States -> Kentucky
6,United States,Kentucky,United States -> Kentucky
7,United States,Georgia,United States -> Georgia
8,United States,Kentucky,United States -> Kentucky
9,United States,Kentucky,United States -> Kentucky


In [14]:
# Create priliminary lead time only when dates are validated
df["Preliminary_Lead_Time"] = (
    df["Ship_Date"] - df["Order_Date"]
).dt.days

In [15]:
print(df["Preliminary_Lead_Time"].describe())

print(
    "Negative lead times:",
    (df["Preliminary_Lead_Time"] < 0).sum()
)

print(
    "Zero lead times:",
    (df["Preliminary_Lead_Time"] == 0).sum()
)

count    10194.000000
mean      1320.841868
std        262.444892
min        904.000000
25%       1271.000000
50%       1274.000000
75%       1638.000000
max       1642.000000
Name: Preliminary_Lead_Time, dtype: float64
Negative lead times: 0
Zero lead times: 0


In [16]:
# Create defensible calender features
df["Order_Year"] = df["Order_Date"].dt.year
df["Order_Month"] = df["Order_Date"].dt.month
df["Order_Quarter"] = df["Order_Date"].dt.quarter
df["Order_Year_Month"] = (
    df["Order_Date"].dt.to_period("M").astype("string")
)

In [17]:
categorical_cols = [
    "Ship Mode",
    "Country/Region",
    "City",
    "State/Province",
    "Division",
    "Region",
    "Product ID",
    "Product Name",
]

for col in categorical_cols:
    df[col] = df[col].astype("string").str.strip()

In [18]:
# Validate the engineering data
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Exact duplicate rows:", df.duplicated().sum())

print(
    df[
        ["Order ID", "Ship Mode", "Region", "State/Province"]
    ].head(10)
)


Rows: 10194
Columns: 28
Exact duplicate rows: 0
                       Order ID       Ship Mode    Region State/Province
0  US-2021-103800-CHO-MIL-31000  Standard Class  Interior          Texas
1  US-2021-112326-CHO-TRI-54000  Standard Class  Interior       Illinois
2  US-2021-112326-CHO-NUT-13000  Standard Class  Interior       Illinois
3  US-2021-112326-CHO-SCR-58000  Standard Class  Interior       Illinois
4  US-2021-141817-CHO-TRI-54000  Standard Class  Atlantic   Pennsylvania
5  US-2021-167199-CHO-SCR-58000  Standard Class      Gulf       Kentucky
6  US-2021-167199-CHO-TRI-54000  Standard Class      Gulf       Kentucky
7  US-2021-106054-CHO-MIL-31000     First Class      Gulf        Georgia
8  US-2021-167199-CHO-NUT-13000  Standard Class      Gulf       Kentucky
9  US-2021-167199-CHO-MIL-31000  Standard Class      Gulf       Kentucky
